# 🎨 Módulo 5: Seaborn

Gráficos estadísticos bonitos en una línea: distribuciones, categorías, relaciones y correlaciones.

**PandasQuest** · versión web interactiva: [https://rosalesluciano.github.io/pandas-quest/](https://rosalesluciano.github.io/pandas-quest/#/m/m05)

**Cómo funciona:**
1. Ejecuta la celda ⚙️ de abajo (carga las tablas y el corrector).
2. En cada ejercicio escribe tu código y guarda la respuesta en `resultado`.
3. Ejecuta la celda: `comprobar()` te dirá si está bien. 🎯

¿Atascado? `ver_solucion("id")` muestra la solución (¡pero inténtalo antes! 😉)

In [ ]:
#@title ⚙️ 1) Ejecuta esta celda primero (carga datos y corrector) { display-mode: "form" }
"""PandasQuest - motor de corrección.

Se usa igual en el navegador (Pyodide), en los notebooks de Colab y en los tests.
Compara el `resultado` del alumno contra el de la solución oficial con tolerancia
a diferencias irrelevantes (tipos int/float, nombres de índice, dtype string, etc.).
"""
import math

import numpy as np
import pandas as pd


def _norm_s(s):
    dt = s.dtype
    if isinstance(dt, pd.CategoricalDtype) or pd.api.types.is_string_dtype(dt):
        s = s.astype(object)
    elif pd.api.types.is_extension_array_dtype(dt):
        if pd.api.types.is_numeric_dtype(dt) and not pd.api.types.is_bool_dtype(dt):
            s = s.astype("float64")
        else:
            s = s.astype(object)
    if s.dtype == object:
        s = s.where(s.notna(), None)
    return s


def _norm_index(idx):
    if isinstance(idx, pd.MultiIndex):
        return idx
    if isinstance(idx, pd.CategoricalIndex) or pd.api.types.is_string_dtype(idx.dtype):
        return pd.Index(list(idx), dtype=object)
    return idx


def _norm(o):
    if isinstance(o, pd.Series):
        o = _norm_s(o.copy())
    else:
        cols = list(o.columns)
        o = pd.DataFrame({i: _norm_s(o.iloc[:, i]) for i in range(o.shape[1])}, index=o.index)
        o.columns = _norm_index(pd.Index(cols))
    o.index = _norm_index(o.index)
    return o


def _sort(o, ignore_index):
    if ignore_index:
        o = o.reset_index(drop=True)
        try:
            if isinstance(o, pd.Series):
                o = o.sort_values(kind="mergesort")
            else:
                o = o.sort_values(by=list(o.columns), kind="mergesort")
        except TypeError:
            key = o.astype(str) if isinstance(o, pd.Series) else o.astype(str).agg("|".join, axis=1)
            o = o.iloc[np.argsort(key.to_numpy(), kind="mergesort")]
        return o.reset_index(drop=True)
    try:
        return o.sort_index()
    except TypeError:
        return o


def _tipo(x):
    nombres = {pd.DataFrame: "un DataFrame", pd.Series: "una Series"}
    return nombres.get(type(x), f"un {type(x).__name__}")


def _scalar_eq(u, e):
    if isinstance(e, (np.generic,)):
        e = e.item()
    if isinstance(u, (np.generic,)):
        u = u.item()
    if isinstance(e, float) or isinstance(u, float):
        try:
            u, e = float(u), float(e)
        except (TypeError, ValueError):
            return False
        if math.isnan(e):
            return math.isnan(u)
        return math.isclose(u, e, rel_tol=1e-6, abs_tol=1e-6)
    if isinstance(e, (list, tuple)) and isinstance(u, (list, tuple, np.ndarray, pd.Index)):
        u = list(u)
        return len(u) == len(e) and all(_scalar_eq(a, b) for a, b in zip(u, e))
    try:
        return bool(u == e)
    except Exception:
        return False


def _array_compare(u, e):
    if isinstance(u, (pd.DataFrame, pd.Series)):
        u = u.to_numpy()
    try:
        ua = np.asarray(u)
    except Exception:
        return False, f"Se esperaba un array de NumPy y tu resultado es {_tipo(u)}."
    if not isinstance(u, np.ndarray) and not isinstance(u, (list, tuple)):
        return False, f"Se esperaba un array de NumPy y tu resultado es {_tipo(u)}."
    if ua.shape != e.shape:
        return False, f"Forma distinta: tu array tiene shape {ua.shape} y se esperaba {e.shape}."
    num = np.issubdtype(e.dtype, np.number) or e.dtype == bool
    try:
        if num and (np.issubdtype(ua.dtype, np.number) or ua.dtype == bool):
            ok = np.allclose(ua.astype(float), e.astype(float), rtol=1e-6, atol=1e-6, equal_nan=True)
        else:
            ok = ua.tolist() == e.tolist()
    except Exception:
        ok = False
    return (True, "") if ok else (False, "La forma es correcta pero los valores no coinciden. Revisa el cálculo.")


def pq_compare(u, e, ordered=True, ignore_index=False):
    """Devuelve (ok, mensaje)."""
    if isinstance(e, np.ndarray):
        return _array_compare(u, e)
    if isinstance(e, (pd.DataFrame, pd.Series)):
        if not isinstance(u, type(e)):
            extra = ""
            if isinstance(e, pd.Series) and isinstance(u, pd.DataFrame):
                extra = " Pista: con una sola columna usa df['col'] (corchetes simples)."
            if isinstance(e, pd.DataFrame) and isinstance(u, pd.Series):
                extra = " Pista: usa doble corchete df[['col']] o .reset_index() / .to_frame()."
            return False, f"Tu resultado es {_tipo(u)}, pero se esperaba {_tipo(e)}.{extra}"
        if isinstance(e, pd.DataFrame):
            uc, ec = [str(c) for c in u.columns], [str(c) for c in e.columns]
            if uc != ec:
                if sorted(uc) == sorted(ec):
                    return False, f"Tienes las columnas correctas pero en otro orden. Se esperaba: {ec}"
                faltan = [c for c in ec if c not in uc]
                sobran = [c for c in uc if c not in ec]
                msg = "Las columnas no coinciden."
                if faltan:
                    msg += f" Faltan: {faltan}."
                if sobran:
                    msg += f" Sobran: {sobran}."
                return False, msg
        if u.shape != e.shape:
            return False, f"Forma distinta: tu resultado tiene {u.shape} y se esperaba {e.shape} (filas, columnas)."
        u2, e2 = _norm(u), _norm(e)
        if not ordered:
            u2, e2 = _sort(u2, ignore_index), _sort(e2, ignore_index)
        elif ignore_index:
            u2, e2 = u2.reset_index(drop=True), e2.reset_index(drop=True)
        kw = dict(check_dtype=False, check_names=False, check_index_type=False,
                  check_exact=False, rtol=1e-6, atol=1e-6)
        try:
            if isinstance(e2, pd.DataFrame):
                pd.testing.assert_frame_equal(u2, e2, check_column_type=False, check_freq=False, **kw)
            else:
                pd.testing.assert_series_equal(u2, e2, check_freq=False, **kw)
        except AssertionError as err:
            txt = str(err)
            if "index" in txt.lower() and "values are different" in txt.lower() and "iloc" not in txt:
                return False, "Los valores parecen correctos pero el índice no coincide. ¿Hace falta ordenar o usar reset_index()?"
            if ordered and not ignore_index:
                try:
                    pd.testing.assert_frame_equal(
                        u2.to_frame() if isinstance(u2, pd.Series) else u2,
                        e2.to_frame() if isinstance(e2, pd.Series) else e2,
                        check_like=True, **kw)
                except AssertionError:
                    pass
                else:
                    return False, "Mismos datos, pero en otro orden. Revisa cómo ordenas las filas."
            return False, "Los valores no coinciden con lo esperado. Revisa la lógica y vuelve a intentarlo."
        return True, ""
    if isinstance(u, (pd.DataFrame, pd.Series)):
        return False, f"Tu resultado es {_tipo(u)}, pero se esperaba un valor simple ({type(e).__name__}). ¿Te falta .iloc[0], .sum() o similar?"
    if _scalar_eq(u, e):
        return True, ""
    return False, f"Tu resultado es {u!r}, pero no es el valor esperado."


# ---------------------------------------------------------------- gráficos
def _plt():
    import matplotlib.pyplot as plt
    return plt


def _fig():
    plt = _plt()
    return plt.gcf() if plt.get_fignums() else None


def _ax(i=0):
    f = _fig()
    if f is None or len(f.axes) <= i:
        raise LookupError("no hay ningún gráfico dibujado")
    return f.axes[i]


def _legend(ax=None):
    ax = ax or _ax()
    lg = ax.get_legend()
    return [t.get_text() for t in lg.get_texts()] if lg else []


def _heights(ax=None):
    return [round(float(p.get_height()), 6) for p in (ax or _ax()).patches]


def _widths(ax=None):
    return [round(float(p.get_width()), 6) for p in (ax or _ax()).patches]


def _xticks(ax=None):
    return [t.get_text() for t in (ax or _ax()).get_xticklabels()]


def _hex(color):
    import matplotlib.colors as mcolors
    return mcolors.to_hex(color)


_HELPERS = {"_fig": _fig, "_ax": _ax, "_legend": _legend, "_heights": _heights,
            "_widths": _widths, "_xticks": _xticks, "_hex": _hex}


def pq_prepare():
    """Deja matplotlib limpio antes de ejecutar código (si está cargado)."""
    import sys
    if "matplotlib.pyplot" in sys.modules:
        plt = sys.modules["matplotlib.pyplot"]
        plt.close("all")
        plt.show = lambda *a, **k: None


def pq_check(ns, get_expected, opts):
    """Evalúa el ejercicio.

    ns = namespace del alumno. get_expected = función que devuelve el namespace de la
    solución (solo se ejecuta si hace falta, para no pisar los gráficos del alumno).
    """
    custom = opts.get("custom")
    if custom:
        if opts.get("plot") and _fig() is None:
            return False, "Todavía no hay ningún gráfico. ¡Dibuja algo! 🎨"
        env = dict(_HELPERS)
        env.update(ns)
        try:
            ok = bool(eval(custom, env))
        except Exception as err:
            return False, f"{opts.get('custom_msg', 'Todavía no es correcto.')} ({type(err).__name__}: {err})"
        return ok, "" if ok else opts.get("custom_msg", "Todavía no es correcto. ¡Sigue intentándolo!")
    if "resultado" not in ns:
        return False, "Guarda tu respuesta en la variable `resultado`."
    exp_ns = get_expected() if callable(get_expected) else get_expected
    return pq_compare(ns["resultado"], exp_ns["resultado"],
                      ordered=opts.get("ordered", True),
                      ignore_index=opts.get("ignore_index", False))


_ERR_TIPS = [
    ("truth value of a Series is ambiguous", "Para combinar condiciones usa & (y), | (o) y pon cada condición entre paréntesis."),
    ("KeyError", "Revisa que el nombre de la columna esté bien escrito (mayúsculas, tildes, espacios)."),
    ("NameError", "Estás usando una variable que no existe. ¿Un error de tipeo?"),
    ("SyntaxError", "Hay un error de sintaxis: revisa paréntesis, comillas y que la línea `resultado = ...` esté completa."),
    ("AttributeError", "Ese método o atributo no existe para este objeto. Revisa el nombre."),
    ("TypeError", "Algún tipo de dato no encaja (¿texto mezclado con números?)."),
    ("IndentationError", "Revisa la sangría (espacios al inicio de la línea)."),
    ("MergeError", "Revisa las columnas que usas para unir (on / left_on / right_on)."),
]


def pq_error_tip(msg):
    for key, tip in _ERR_TIPS:
        if key in msg:
            return tip
    return ""


_SETUP = 'import pandas as pd\nimport numpy as np\n\nclientes = pd.DataFrame({\n    "cliente_id": range(1, 13),\n    "nombre": ["Ana", "Bruno", "Carla", "Diego", "Elena", "Facundo",\n               "Gabriela", "Hugo", "Inés", "Julián", "Karina", "Lucas"],\n    "ciudad": ["Madrid", "Buenos Aires", "CDMX", "Bogotá", "Madrid", "Lima",\n               "Buenos Aires", "CDMX", "Santiago", "Bogotá", "Madrid", "Lima"],\n    "pais": ["España", "Argentina", "México", "Colombia", "España", "Perú",\n             "Argentina", "México", "Chile", "Colombia", "España", "Perú"],\n    "edad": [34, 28, 45, 23, 51, 39, 30, 27, 42, 36, 25, 48],\n    "segmento": ["Premium", "Básico", "Premium", "Básico", "Premium", "Estándar",\n                 "Estándar", "Básico", "Premium", "Estándar", "Básico", "Estándar"],\n    "email": ["ana@mail.com", None, "carla@mail.com", "diego@mail.com", None, "facu@mail.com",\n              "gabi@mail.com", None, "ines@mail.com", "julian@mail.com", "kari@mail.com", "lucas@mail.com"],\n    "fecha_registro": pd.to_datetime([\n        "2023-01-15", "2023-02-03", "2023-02-20", "2023-03-11", "2023-04-02", "2023-05-19",\n        "2023-06-07", "2023-07-23", "2023-08-30", "2023-09-14", "2023-10-05", "2023-11-28"]),\n})\n\nproductos = pd.DataFrame({\n    "producto_id": range(1, 11),\n    "producto": ["Laptop", "Mouse", "Teclado", "Monitor", "Auriculares",\n                 "Webcam", "Silla gamer", "Escritorio", "Tablet", "Micrófono"],\n    "categoria": ["Computación", "Accesorios", "Accesorios", "Computación", "Audio",\n                  "Accesorios", "Muebles", "Muebles", "Computación", "Audio"],\n    "precio": [1200.0, 25.5, 45.0, 300.0, 80.0, 60.0, 250.0, 180.0, 450.0, 95.0],\n    "stock": [5, 150, 80, 20, 60, 0, 12, 7, 15, 0],\n})\n\n_estados = ["entregado", "enviado", "entregado", "cancelado", "entregado", "pendiente"]\npedidos = pd.DataFrame({\n    "pedido_id": range(1001, 1031),\n    "cliente_id": [(i * 7) % 11 + 1 for i in range(29)] + [99],\n    "producto_id": [(i * 3) % 10 + 1 for i in range(30)],\n    "cantidad": [(i % 4) + 1 for i in range(30)],\n    "fecha": pd.date_range("2024-01-03", periods=30, freq="5D"),\n    "estado": [_estados[i % 6] for i in range(30)],\n})\n\nempleados = pd.DataFrame({\n    "emp_id": range(1, 11),\n    "nombre": ["Marta", "Pablo", "Sofía", "Tomás", "Valeria",\n               "Andrés", "Lucía", "Martín", "Paula", "Ramiro"],\n    "depto": ["Dirección", "Ventas", "Ventas", "Ventas", "IT",\n              "IT", "IT", "Marketing", "Marketing", "Ventas"],\n    "salario": [9000, 4200, 3900, 4000, 5200, 4800, 6100, 3700, 4100, 3500],\n    "jefe_id": pd.array([None, 1, 2, 2, 1, 5, 5, 1, 8, 2], dtype="Int64"),\n    "fecha_ingreso": pd.to_datetime([\n        "2015-03-01", "2018-06-15", "2019-01-10", "2020-09-01", "2017-11-20",\n        "2021-02-01", "2016-08-08", "2022-04-18", "2019-07-07", "2023-01-09"]),\n})\n\n_i = np.arange(90)\n_f = pd.date_range("2024-01-01", periods=90, freq="D")\n_v = 1000 + 8 * _i + 250 * (_f.dayofweek >= 5) + ((_i * 37) % 11) * 20\n_v = np.where(np.isin(_i, [20, 55, 77]), _v + 1500, _v)\nventas_diarias = pd.DataFrame({\n    "fecha": _f,\n    "ventas": _v.astype(float),\n    "visitas": (300 + (_i * 13) % 97 + 2 * _i).astype(int),\n})\n\nsucio = pd.DataFrame({\n    "id": [1, 2, 2, 3, 4, 5, 6, 7],\n    "nombre": ["  ana ", "BRUNO", "BRUNO", "carla", None, "Diego  ", "elena", "facundo"],\n    "edad": ["34", "28", "28", "n/d", "23", "51", None, "39"],\n    "ciudad": ["madrid", "Buenos Aires", "Buenos Aires", " santiago", "bogotá", "Madrid ", "madrid", "LIMA"],\n    "monto": ["$1,200.50", "$300", "$300", "$45.00", None, "$80.5", "$99", "$1,000"],\n})\ndel _i, _f, _v, _estados\n\nimport matplotlib.pyplot as plt\nimport seaborn as sns\n\n_rng = np.random.default_rng(7)\n_n = 120\npropinas = pd.DataFrame({\n    "cuenta": np.round(_rng.gamma(4, 5, _n) + 5, 2),\n    "dia": _rng.choice(["Jue", "Vie", "Sáb", "Dom"], _n, p=[0.2, 0.2, 0.35, 0.25]),\n    "momento": _rng.choice(["Almuerzo", "Cena"], _n, p=[0.35, 0.65]),\n    "fumador": _rng.choice(["Sí", "No"], _n, p=[0.4, 0.6]),\n    "personas": _rng.integers(1, 7, _n),\n})\npropinas["propina"] = np.round(propinas["cuenta"] * _rng.uniform(0.08, 0.22, _n) + 0.5, 2)\ndel _rng, _n\n'
_SOL_B64 = 'eyJzbnMwMSI6IHsic29sdXRpb24iOiAic25zLmhpc3RwbG90KGRhdGE9cHJvcGluYXMsIHg9J2N1ZW50YScpIiwgImNoZWNrIjogeyJwbG90IjogdHJ1ZSwgImN1c3RvbSI6ICInaGlzdHBsb3QnIGluIF9wcV9jb2RlIGFuZCBsZW4oX2F4KCkucGF0Y2hlcykgPiAwIiwgImN1c3RvbV9tc2ciOiAiVXNhIHNucy5oaXN0cGxvdCBjb24gbGEgY29sdW1uYSAnY3VlbnRhJy4ifSwgInhwIjogMTB9LCAic25zMDIiOiB7InNvbHV0aW9uIjogInNucy5oaXN0cGxvdChkYXRhPXByb3BpbmFzLCB4PSdwcm9waW5hJywgYmlucz0xNSwga2RlPVRydWUpIiwgImNoZWNrIjogeyJwbG90IjogdHJ1ZSwgImN1c3RvbSI6ICJsZW4oX2F4KCkucGF0Y2hlcykgPT0gMTUgYW5kIGxlbihfYXgoKS5saW5lcykgPj0gMSIsICJjdXN0b21fbXNnIjogIkRlYmVuIHZlcnNlIDE1IGJhcnJhcyB5IGxhIGN1cnZhIEtERS4ifSwgInhwIjogMjB9LCAic25zMDMiOiB7InNvbHV0aW9uIjogInNucy5rZGVwbG90KGRhdGE9cHJvcGluYXMsIHg9J2N1ZW50YScsIGh1ZT0nbW9tZW50bycsIGZpbGw9VHJ1ZSkiLCAiY2hlY2siOiB7InBsb3QiOiB0cnVlLCAiY3VzdG9tIjogIidrZGVwbG90JyBpbiBfcHFfY29kZSBhbmQgc29ydGVkKF9sZWdlbmQoKSkgPT0gWydBbG11ZXJ6bycsICdDZW5hJ10iLCAiY3VzdG9tX21zZyI6ICJVc2Egc25zLmtkZXBsb3QgY29uIGh1ZT0nbW9tZW50bycgKGxhIGxleWVuZGEgZGViZSBtb3N0cmFyIEFsbXVlcnpvIHkgQ2VuYSkuIn0sICJ4cCI6IDIwfSwgInNuczA0IjogeyJzb2x1dGlvbiI6ICJzbnMuc2NhdHRlcnBsb3QoZGF0YT1wcm9waW5hcywgeD0nY3VlbnRhJywgeT0ncHJvcGluYScsIGh1ZT0nbW9tZW50bycpIiwgImNoZWNrIjogeyJwbG90IjogdHJ1ZSwgImN1c3RvbSI6ICInc2NhdHRlcnBsb3QnIGluIF9wcV9jb2RlIGFuZCBzdW0obGVuKGMuZ2V0X29mZnNldHMoKSkgZm9yIGMgaW4gX2F4KCkuY29sbGVjdGlvbnMpID09IDEyMCBhbmQgeydBbG11ZXJ6bycsICdDZW5hJ30gPD0gc2V0KF9sZWdlbmQoKSkiLCAiY3VzdG9tX21zZyI6ICJEZWJlbiB2ZXJzZSBsb3MgMTIwIHB1bnRvcywgY29sb3JlYWRvcyBwb3IgJ21vbWVudG8nLiJ9LCAieHAiOiAyMH0sICJzbnMwNSI6IHsic29sdXRpb24iOiAic25zLnJlZ3Bsb3QoZGF0YT1wcm9waW5hcywgeD0nY3VlbnRhJywgeT0ncHJvcGluYScpIiwgImNoZWNrIjogeyJwbG90IjogdHJ1ZSwgImN1c3RvbSI6ICIncmVncGxvdCcgaW4gX3BxX2NvZGUgYW5kIGxlbihfYXgoKS5saW5lcykgPj0gMSBhbmQgbGVuKF9heCgpLmNvbGxlY3Rpb25zKSA+PSAxIiwgImN1c3RvbV9tc2ciOiAiVXNhIHNucy5yZWdwbG90IGNvbiBjdWVudGEgZW4gWCB5IHByb3BpbmEgZW4gWS4ifSwgInhwIjogMzV9LCAic25zMDYiOiB7InNvbHV0aW9uIjogInNucy5zZXRfdGhlbWUoc3R5bGU9J3doaXRlZ3JpZCcpXG5zbnMubGluZXBsb3QoZGF0YT12ZW50YXNfZGlhcmlhcywgeD0nZmVjaGEnLCB5PSd2ZW50YXMnKVxucGx0LnRpdGxlKCdUZW5kZW5jaWEgZGUgdmVudGFzJylcbnBsdC54dGlja3Mocm90YXRpb249NDUpIiwgImNoZWNrIjogeyJwbG90IjogdHJ1ZSwgImN1c3RvbSI6ICInbGluZXBsb3QnIGluIF9wcV9jb2RlIGFuZCAnd2hpdGVncmlkJyBpbiBfcHFfY29kZSBhbmQgbGVuKF9heCgpLmxpbmVzKSA+PSAxIGFuZCBfYXgoKS5nZXRfdGl0bGUoKSA9PSAnVGVuZGVuY2lhIGRlIHZlbnRhcyciLCAiY3VzdG9tX21zZyI6ICJVc2Egc2V0X3RoZW1lKHN0eWxlPSd3aGl0ZWdyaWQnKSwgc25zLmxpbmVwbG90IHkgZWwgdMOtdHVsbyAnVGVuZGVuY2lhIGRlIHZlbnRhcycuIn0sICJ4cCI6IDIwfSwgInNuczA3IjogeyJzb2x1dGlvbiI6ICJzbnMuYm94cGxvdChkYXRhPXByb3BpbmFzLCB4PSdkaWEnLCB5PSdwcm9waW5hJywgb3JkZXI9WydKdWUnLCAnVmllJywgJ1PDoWInLCAnRG9tJ10pIiwgImNoZWNrIjogeyJwbG90IjogdHJ1ZSwgImN1c3RvbSI6ICInYm94cGxvdCcgaW4gX3BxX2NvZGUgYW5kIF94dGlja3MoKSA9PSBbJ0p1ZScsICdWaWUnLCAnU8OhYicsICdEb20nXSIsICJjdXN0b21fbXNnIjogIlVzYSBzbnMuYm94cGxvdCBjb24geD0nZGlhJywgeT0ncHJvcGluYScgeSBlbCBvcmRlbiBkZSBkw61hcyBpbmRpY2Fkby4ifSwgInhwIjogMjB9LCAic25zMDgiOiB7InNvbHV0aW9uIjogInNucy5iYXJwbG90KGRhdGE9cHJvcGluYXMsIHg9J2RpYScsIHk9J3Byb3BpbmEnLCBvcmRlcj1bJ0p1ZScsICdWaWUnLCAnU8OhYicsICdEb20nXSwgZXJyb3JiYXI9Tm9uZSkiLCAiY2hlY2siOiB7InBsb3QiOiB0cnVlLCAiY3VzdG9tIjogIidiYXJwbG90JyBpbiBfcHFfY29kZSBhbmQgbGVuKF9heCgpLnBhdGNoZXMpID09IDQgYW5kIG5wLmFsbGNsb3NlKF9oZWlnaHRzKCksIHByb3BpbmFzLmdyb3VwYnkoJ2RpYScpWydwcm9waW5hJ10ubWVhbigpLnJlaW5kZXgoWydKdWUnLCAnVmllJywgJ1PDoWInLCAnRG9tJ10pLnZhbHVlcykiLCAiY3VzdG9tX21zZyI6ICJEZWJlbiB2ZXJzZSA0IGJhcnJhcyBjb24gbGEgcHJvcGluYSBtZWRpYSBkZSBjYWRhIGTDrWEsIGVuIGVsIG9yZGVuIHBlZGlkby4ifSwgInhwIjogMzV9LCAic25zMDkiOiB7InNvbHV0aW9uIjogInNucy5jb3VudHBsb3QoZGF0YT1wcm9waW5hcywgeD0nbW9tZW50bycpIiwgImNoZWNrIjogeyJwbG90IjogdHJ1ZSwgImN1c3RvbSI6ICInY291bnRwbG90JyBpbiBfcHFfY29kZSBhbmQgbGVuKF9heCgpLnBhdGNoZXMpID09IDIgYW5kIHN1bShfaGVpZ2h0cygpKSA9PSAxMjAiLCAiY3VzdG9tX21zZyI6ICJVc2Egc25zLmNvdW50cGxvdCBjb24geD0nbW9tZW50bycgKDIgYmFycmFzIHF1ZSBzdW1lbiAxMjApLiJ9LCAieHAiOiAxMH0sICJzbnMxMCI6IHsic29sdXRpb24iOiAic25zLmJveHBsb3QoZGF0YT1wcm9waW5hcywgeD0nZGlhJywgeT0ncHJvcGluYScsIGh1ZT0nZnVtYWRvcicsIG9yZGVyPVsnSnVlJywgJ1ZpZScsICdTw6FiJywgJ0RvbSddKVxucGx0LnRpdGxlKCdQcm9waW5hcyBwb3IgZMOtYSB5IGZ1bWFkb3InKSIsICJjaGVjayI6IHsicGxvdCI6IHRydWUsICJjdXN0b20iOiAiJ2JveHBsb3QnIGluIF9wcV9jb2RlIGFuZCBfeHRpY2tzKCkgPT0gWydKdWUnLCAnVmllJywgJ1PDoWInLCAnRG9tJ10gYW5kIHNvcnRlZChfbGVnZW5kKCkpID09IFsnTm8nLCAnU8OtJ10gYW5kIF9heCgpLmdldF90aXRsZSgpID09ICdQcm9waW5hcyBwb3IgZMOtYSB5IGZ1bWFkb3InIiwgImN1c3RvbV9tc2ciOiAiQm94cGxvdCBwb3IgZMOtYSAob3JkZW4gcGVkaWRvKSwgaHVlPSdmdW1hZG9yJyB5IGVsIHTDrXR1bG8gZXhhY3RvLiJ9LCAieHAiOiAzNX0sICJzbnMxMSI6IHsic29sdXRpb24iOiAiY29yciA9IHByb3BpbmFzW1snY3VlbnRhJywgJ3Byb3BpbmEnLCAncGVyc29uYXMnXV0uY29ycigpXG5zbnMuaGVhdG1hcChjb3JyLCBhbm5vdD1UcnVlLCBjbWFwPSdjb29sd2FybScpIiwgImNoZWNrIjogeyJwbG90IjogdHJ1ZSwgImN1c3RvbSI6ICInaGVhdG1hcCcgaW4gX3BxX2NvZGUgYW5kICdjb29sd2FybScgaW4gX3BxX2NvZGUgYW5kIGxlbihfYXgoKS50ZXh0cykgPT0gOSIsICJjdXN0b21fbXNnIjogIkVsIGhlYXRtYXAgZGViZSBzZXIgZGUgM3gzLCBjb24gYW5ub3Q9VHJ1ZSB5IGNtYXA9J2Nvb2x3YXJtJy4ifSwgInhwIjogMzV9fQ=='
_TABLAS = ['propinas', 'ventas_diarias']

import base64 as _b64, json as _json, re as _re, warnings as _w
_w.simplefilter("ignore")
_SOL = _json.loads(_b64.b64decode(_SOL_B64).decode("utf-8"))

def reiniciar_datos():
    """Vuelve a cargar todas las tablas limpias (cada ejercicio empieza de cero)."""
    globals().pop("resultado", None)
    exec(_SETUP, globals())
    pq_prepare()

def _codigo_celda():
    try:
        src = get_ipython().history_manager.input_hist_raw[-1]
    except Exception:
        return ""
    src = "\n".join(l for l in src.splitlines() if "comprobar(" not in l)
    return "\n".join(_re.sub(r"#.*$", "", l) for l in src.splitlines())

def comprobar(ej):
    info = _SOL[ej]
    def esperado():
        exp_ns = {}
        exec(_SETUP, exp_ns)
        exec(info["solution"], exp_ns)
        return exp_ns
    g = globals()
    g["_pq_code"] = _codigo_celda()
    ok, msg = pq_check(g, esperado, info["check"])
    if ok:
        from IPython.display import HTML, display
        display(HTML('<div style="padding:12px 16px;border-radius:12px;background:#0f9d6e;color:#fff;'
                     'font:600 15px system-ui">🎉 ¡Correcto! +' + str(info["xp"]) + ' XP · Sigue así 🐼</div>'))
    else:
        from IPython.display import HTML, display
        import html as _h
        display(HTML('<div style="padding:12px 16px;border-radius:12px;background:#b4235a;color:#fff;'
                     'font:500 15px system-ui">❌ Todavía no. ' + _h.escape(msg) + '</div>'))
    if "resultado" in g:
        return g["resultado"]

def ver_solucion(ej):
    print(_SOL[ej]["solution"])

reiniciar_datos()
print("✅ Datos cargados:", ", ".join(_TABLAS))


---
# 📌 Distribuciones

## 1. Hola, seaborn
`Básico` · **+10 XP**

**Seaborn** está construida sobre matplotlib y está pensada para **gráficos estadísticos**: con una sola línea obtienes gráficos bonitos que entienden DataFrames directamente.

```python
import seaborn as sns
sns.histplot(data=df, x="columna")
```

El patrón es siempre el mismo:
- `data=`: el DataFrame;
- `x=` / `y=`: los **nombres** de las columnas (como texto);
- `hue=`: una columna para **colorear por grupos** (lo verás enseguida).

En este módulo tienes el dataset `propinas`: las cuentas de un restaurante, con `cuenta`, `propina`, `dia`, `momento` (Almuerzo/Cena), `fumador` y `personas`.

> 💡 Como seaborn dibuja con matplotlib, puedes seguir usando `plt.title()`, `plt.xlabel()`…

### 🎯 Tu misión
Dibuja un histograma de la columna `cuenta` de `propinas` con `sns.histplot`.

<details><summary>💡 Pista</summary>

`sns.histplot(data=propinas, x='cuenta')`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

# sns y plt ya están importados


comprobar("sns01")

## 2. Histograma con curva de densidad
`Intermedio` · **+20 XP**

`histplot` tiene parámetros muy útiles:

- `bins=`: cantidad de barras;
- `kde=True`: añade una **curva de densidad** suavizada, que muestra la forma de la distribución;
- `stat="percent"`: muestra porcentajes en lugar de conteos.

```python
sns.histplot(data=df, x="edad", bins=20, kde=True)
```

> 💼 **En el trabajo:** antes de modelar, se mira la distribución de cada variable. Una variable muy **asimétrica** (con cola larga, como los salarios) a veces se transforma con un logaritmo.

### 🎯 Tu misión
Dibuja el histograma de `propina` con **15 bins** y la curva de densidad (`kde=True`).

<details><summary>💡 Pista</summary>

`bins=15, kde=True`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

sns.histplot(data=propinas, x='propina')

comprobar("sns02")

## 3. Comparar distribuciones con hue
`Intermedio` · **+20 XP**

El parámetro **`hue`** colorea según una columna categórica y dibuja **un grupo por color**, con su leyenda automática. Es la superpotencia de seaborn.

`kdeplot` dibuja solo la curva de densidad, así que es ideal para **superponer grupos** sin que las barras se tapen:

```python
sns.kdeplot(data=df, x="salario", hue="genero", fill=True)
```

`fill=True` rellena el área bajo cada curva.

### 🎯 Tu misión
Compara la distribución de `cuenta` entre **Almuerzo** y **Cena**: usa `sns.kdeplot` con `hue='momento'` y `fill=True`.

<details><summary>💡 Pista</summary>

`sns.kdeplot(data=propinas, x='cuenta', hue='momento', fill=True)`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios



comprobar("sns03")

---
# 📌 Relaciones

## 4. Dispersión con colores
`Intermedio` · **+20 XP**

`sns.scatterplot` es el gráfico de dispersión de seaborn. Con `hue` coloreas los puntos por grupo y con `size` puedes hacer que su tamaño dependa de otra variable:

```python
sns.scatterplot(data=df, x="m2", y="precio", hue="barrio")
```

De un vistazo ves si la relación entre X e Y cambia según el grupo.

### 🎯 Tu misión
Dibuja un scatter de `propinas` con `cuenta` en X, `propina` en Y y el color según `momento`.

<details><summary>💡 Pista</summary>

`sns.scatterplot(data=propinas, x='cuenta', y='propina', hue='momento')`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios



comprobar("sns04")

## 5. Línea de tendencia
`Avanzado` · **+35 XP**

`sns.regplot` dibuja los puntos **y una recta de regresión** que resume la tendencia. La franja sombreada es el **intervalo de confianza**: cuanto más estrecha, más segura es la tendencia.

```python
sns.regplot(data=df, x="horas_estudio", y="nota")
```

Es tu primer contacto con un **modelo**: la recta que mejor se ajusta a los puntos. En el módulo de Machine Learning la calcularás tú mismo con scikit-learn.

### 🎯 Tu misión
Dibuja la relación entre `cuenta` (X) y `propina` (Y) con su recta de tendencia, usando `sns.regplot`.

<details><summary>💡 Pista</summary>

`sns.regplot(data=propinas, x='cuenta', y='propina')`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios



comprobar("sns05")

## 6. Tendencias en el tiempo
`Intermedio` · **+20 XP**

`sns.lineplot` dibuja series temporales y, si hay varias observaciones por punto, calcula la media y su intervalo automáticamente.

Con `sns.set_theme(style=...)` cambias el estilo de **todos** los gráficos siguientes: `"whitegrid"`, `"darkgrid"`, `"white"`, `"ticks"`.

```python
sns.set_theme(style="whitegrid")
sns.lineplot(data=df, x="fecha", y="ventas")
plt.xticks(rotation=45)
```

### 🎯 Tu misión
Aplica el estilo **"whitegrid"**, dibuja con `sns.lineplot` las `ventas` por `fecha` de `ventas_diarias` y pon el título **"Tendencia de ventas"**.

<details><summary>💡 Pista</summary>

`sns.set_theme(style='whitegrid'); sns.lineplot(...); plt.title('...')`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios



comprobar("sns06")

---
# 📌 Comparar categorías

## 7. Boxplot
`Intermedio` · **+20 XP**

El **boxplot** (diagrama de caja) resume una distribución en 5 números:

- la **línea central** es la mediana;
- la **caja** va del percentil 25 al 75 (ahí está el 50% central de los datos);
- los **bigotes** llegan hasta los valores "normales";
- los **puntos sueltos** son **outliers** (valores atípicos).

```python
sns.boxplot(data=df, x="categoria", y="precio", order=["A", "B", "C"])
```

`order=` fija el orden de las categorías en el eje.

> 💼 **En el trabajo:** comparar sueldos por departamento o tiempos de entrega por proveedor, y detectar outliers de un vistazo.

### 🎯 Tu misión
Dibuja un boxplot de `propina` por `dia`, con los días en el orden `['Jue', 'Vie', 'Sáb', 'Dom']`.

<details><summary>💡 Pista</summary>

`sns.boxplot(data=propinas, x='dia', y='propina', order=['Jue', 'Vie', 'Sáb', 'Dom'])`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios



comprobar("sns07")

## 8. Barras con promedios
`Avanzado` · **+35 XP**

`sns.barplot` **calcula por ti** la media de `y` para cada categoría de `x` (no cuenta filas: promedia).

```python
sns.barplot(data=df, x="region", y="ventas", errorbar=None)
```

Por defecto dibuja una **barra de error** (el intervalo de confianza de la media); con `errorbar=None` la quitas. Con `estimator="sum"` suma en vez de promediar.

> ⚠️ No lo confundas con `countplot`, que **cuenta filas** (lo verás en el siguiente ejercicio).

### 🎯 Tu misión
Dibuja un barplot con la **propina media** por `dia` (orden `['Jue', 'Vie', 'Sáb', 'Dom']`) y sin barras de error.

<details><summary>💡 Pista</summary>

`sns.barplot(data=propinas, x='dia', y='propina', order=['Jue', 'Vie', 'Sáb', 'Dom'], errorbar=None)`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios



comprobar("sns08")

## 9. Contar categorías
`Básico` · **+10 XP**

`sns.countplot` dibuja una barra por categoría con **cuántas filas** tiene cada una. Es la versión gráfica de `value_counts()`.

```python
sns.countplot(data=df, x="metodo_pago")
```

Con `hue` puedes partir cada barra por otra variable, por ejemplo `hue="fumador"`.

### 🎯 Tu misión
Dibuja con `sns.countplot` cuántas cuentas hay de cada `momento` (Almuerzo/Cena).

<details><summary>💡 Pista</summary>

`sns.countplot(data=propinas, x='momento')`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios



comprobar("sns09")

## 10. Dos variables categóricas
`Avanzado` · **+35 XP**

Combinando `x` y `hue` en un mismo gráfico comparas **dos variables categóricas a la vez**. Por ejemplo, un boxplot por día que además separa fumadores y no fumadores:

```python
sns.boxplot(data=df, x="dia", y="propina", hue="fumador")
plt.title("...")
```

Cada día tendrá **dos cajas**, una por grupo, y la leyenda indica cuál es cuál.

### 🎯 Tu misión
Dibuja un boxplot de `propina` por `dia` (orden `['Jue', 'Vie', 'Sáb', 'Dom']`), separado por `fumador` con `hue`, y pon el título **"Propinas por día y fumador"**.

<details><summary>💡 Pista</summary>

`sns.boxplot(..., hue='fumador') + plt.title('Propinas por día y fumador')`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios



comprobar("sns10")

---
# 📌 Correlación

## 11. Mapa de calor de correlaciones
`Avanzado` · **+35 XP**

La **correlación** mide de -1 a 1 cuánto se mueven juntas dos variables numéricas:
- **1**: suben juntas perfectamente;
- **0**: no hay relación lineal;
- **-1**: cuando una sube, la otra baja.

`df.corr()` calcula la **matriz de correlación** y `sns.heatmap` la pinta con colores. Con `annot=True` escribe el valor en cada celda:

```python
corr = df[["a", "b", "c"]].corr()
sns.heatmap(corr, annot=True, cmap="coolwarm")
```

> 💼 **En el trabajo:** antes de entrenar un modelo, el heatmap de correlaciones muestra qué variables tienen relación con lo que quieres predecir.

> ⚠️ Correlación **no implica** causalidad.

### 🎯 Tu misión
Calcula la correlación entre `cuenta`, `propina` y `personas` y dibújala con `sns.heatmap`, con los valores escritos (`annot=True`) y el mapa de colores `"coolwarm"`.

<details><summary>💡 Pista</summary>

`corr = propinas[['cuenta', 'propina', 'personas']].corr(); sns.heatmap(corr, annot=True, cmap='coolwarm')`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

corr = 


comprobar("sns11")

---
# 🏁 ¡Módulo completado!
Vuelve a [PandasQuest](https://rosalesluciano.github.io/pandas-quest/) para sumar tus XP y seguir con el siguiente módulo. 🚀